# Week 1 Homework

In [1]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper_hw import RAGBase
from openai import OpenAI
from minsearch import Index

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [4]:
len(documents)

72

### Q2. Indexing and searching

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [7]:
index = build_index(documents)

In [8]:
index

In [9]:
search_results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5,
)

In [10]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

### Q3. RAG

In [ ]:
openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [12]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [13]:
answer

('The agentic loop keeps calling the model with a `while True` loop, then checks whether the model returned any `function_call` items.\n\n- If there are function calls, it runs the tool, appends the tool output to `messages`, and loops again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls this turn**.',
 ResponseUsage(input_tokens=7036, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7125))

### Q4. Chunking

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

### Q5. RAG with chunking

In [16]:
chunk_assistant = RAGBase(
    index=build_index(chunks),
    llm_client=openai_client,
)

In [17]:
chunk_assistant.rag("How does the agentic loop keep calling the model until it stops?")

('It keeps calling the model inside a `while True` loop and checks whether any `function_call` items were returned.\n\n- If there is a function call, the code runs the tool, appends the result, and loops again.\n- If there are no function calls in that turn, it breaks out of the loop and stops.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words: the loop continues until the model returns a final message with no more tool calls.',
 ResponseUsage(input_tokens=2221, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2333))

### Q6. Turning it into an agent

In [19]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [20]:
chunk_index = build_index(chunks)

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
    )

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [26]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG??",
    callback=callback,
)

-> Response received


-> Response received
